# Pagsusuri ng Pag-angkin ng Gastos

Ipinapakita ng notebook na ito kung paano gumawa ng mga ahente na gumagamit ng mga plugin para iproseso ang mga gastusin sa paglalakbay mula sa mga lokal na larawan ng resibo, lumikha ng email para sa pag-angkin ng gastos, at ipakita ang datos ng gastos gamit ang pie chart. Ang mga ahente ay pumipili ng mga function nang dynamic base sa konteksto ng gawain.

Mga Hakbang:
1. Pinoproseso ng OCR Agent ang lokal na larawan ng resibo at kinukuha ang datos ng gastusing panglakbay.
2. Lumilikha ang Email Agent ng isang email para sa pag-angkin ng gastos.

### Halimbawa ng isang sitwasyon ng gastusin sa paglalakbay:
Isipin na ikaw ay isang empleyado na naglalakbay para sa isang pulong ng negosyo sa ibang lungsod. May patakaran ang iyong kumpanya na i-reimburse ang lahat ng makatwirang gastusin na may kinalaman sa paglalakbay. Narito ang paghahati ng mga posibleng gastusin sa paglalakbay:
- Transportasyon:
Pamasahe sa eroplano para sa pabalik-balik na biyahe mula sa iyong lungsod ng tahanan patungo sa lungsod ng destinasyon.
Taxi o mga serbisyo ng ride-hailing papunta at pabalik mula sa paliparan.
Lokal na transportasyon sa lungsod ng destinasyon (tulad ng pampublikong sasakyan, paupahang sasakyan, o taxi).

- Panuluyan:
Pamamalagi sa hotel ng tatlong gabi sa isang mid-range na hotel pang-negosyo malapit sa lugar ng pulong.

- Pagkain:
Pang-araw-araw na allowance para sa pagkain ng almusal, tanghalian, at hapunan, batay sa patakaran ng kumpanya sa per diem.

- Iba pang Gastusin:
Bayad sa paradahan sa paliparan.
Singil sa paggamit ng internet sa hotel.
Tip o maliliit na bayad sa serbisyo.

- Dokumentasyon:
Isusumite mo ang lahat ng resibo (lipad, taxi, hotel, pagkain, atbp.) at isang kumpletong ulat ng gastos para sa reimbursement.


## I-import ang mga kinakailangang librarya

I-import ang mga kinakailangang librarya at mga module para sa notebook.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Tukuyin ang mga Modelo ng Gastos

 Gumawa ng Pydantic na modelo para sa mga indibidwal na gastos at isang ExpenseFormatter na klase upang i-convert ang query ng user sa istrukturadong datos ng gastos.

 Ang bawat gastos ay ire-representa sa pormat:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Pagde-define ng Mga Kasangkapan - Pagbuo ng Email

Gumawa ng isang tool function para bumuo ng email para magsumite ng claim sa gastos.
- Ang tool na ito ay gumagamit ng `@tool` decorator mula sa Microsoft Agent Framework.
- Kinakalkula nito ang kabuuang halaga ng mga gastos at iniaayos ang mga detalye sa isang katawan ng email.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Kasangkapan para sa Pagkuha ng Gastos sa Paglalakbay mula sa mga Larawan ng Resibo

Gumawa ng isang kasangkapan na function upang kunin ang gastos sa paglalakbay mula sa mga larawan ng resibo.
- Ginagamit ng kasangkapang ito ang `@tool` decorator mula sa Microsoft Agent Framework.
- Binabasa nito ang larawan ng resibo, kino-code ito bilang base64, at ibinabalik ang data URI para sa pagsusuri ng agent.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Pagpoproseso ng mga Gastusin

Tukuyin ang mga ahente at ikonekta sila sa isang sunud-sunod na workflow gamit ang `WorkflowBuilder`.
- Kinukuha ng OCR agent ang nakaestrukturang datos ng gastusin mula sa larawan ng resibo gamit ang tool na `load_receipt_image`.
- Kinuha ng Email agent ang na-extract na datos at gumawa ng propesyonal na email para sa claim ng gastusin gamit ang tool na `generate_expense_email`.
- Ang `WorkflowBuilder` gamit ang `add_edge` ay lumilikha ng sunud-sunod na pipeline: OCR Agent → Email Agent.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Pangunahing function

Buuhin ang sunud-sunod na workflow at patakbuhin ito upang iproseso ang imahe ng resibo at makabuo ng email para sa pag-angkin ng gastos.


> **Tandaan:** Ang workflow na ito ay kasalukuyang nagpapasa ng litrato ng resibo bilang base64 na teksto, na hindi ituturing ng karamihan sa mga modelo ng chat (kabilang ang gpt-5-mini) bilang isang larawan.
> Maari rin itong lumampas sa context window ng modelo. Mas mainam na gamitin ang OCR gamit ang Azure AI Vision (o ibang OCR na kasangkapan) at ipasa lamang ang nakuha na teksto, o baguhin upang ipadala ang larawan bilang isang `image_url` na mensahe.
> Kung nais mo lamang iwasan ang mga error sa konteksto, subukan ang mas maliit na larawan ng resibo o isang modelo na may mas malaking context window.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
